# 02 · Data Cleaning & Feature Engineering

**Input:** `data/raw/sales_raw.csv` + the issue list from notebook 01
**Output:** `data/processed/sales_cleaned.csv` and `data/processed/cleaning_log.csv`

## The rules we follow

1. **Fix before you delete.** If a value can be rebuilt reliably from another column, rebuild it.
2. **Delete only what cannot be trusted *and* cannot be rebuilt**, and write down how many rows and why.
3. **Never invent money.** Missing `Sales`/`Profit` is *not* filled with an average — a made-up number would flow straight into every KPI.
4. **Unusual ≠ wrong.** Very large orders are flagged, not removed.
5. **Order matters.** Text is standardised *before* checking duplicates, so `" West"` vs `"West"` cannot hide a duplicate.

All logic lives in `src/data_cleaning.py`. This notebook runs each step **one at a time** so you can see its effect. The automated pipeline (`python src/pipeline.py`) runs exactly the same functions.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

import data_cleaning as dc
from feature_engineering import engineer_features
from utils import RAW_FILE, CLEAN_FILE, CLEANING_LOG_FILE

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

raw = pd.read_csv(RAW_FILE)
log = dc.CleaningLog()          # records every step for the before/after report
log.add("Load raw data", len(raw), len(raw), 0, "Starting point.")
print(f"Raw rows: {len(raw):,}")

Raw rows: 11,567


---
## Step 1 — Strip leading/trailing spaces
**Problem:** `"  Boston "` and `"Boston"` are different values to a computer.
**Decision:** trim every text column. This never loses information — spaces at the edges carry no meaning.

In [2]:
df = dc.strip_whitespace(raw, log)
print("Values changed:", log.entries[-1]["values_changed"])

Values changed: 67


## Step 2 — Standardise `Region` (and rebuild missing ones)
**Problem:** 35 blanks plus spellings such as `east`, `WEST`, `Centrl`.
**Options considered:**

| Option | Verdict |
|---|---|
| Delete rows with bad/missing region | ❌ Loses real sales for a fixable problem |
| Label them "Unknown" | ❌ Sales would be missing from their true region's total |
| **Rebuild from `State`** | ✅ Notebook 01 proved every state belongs to exactly one region |

**Decision:** fix casing/typos, then set `Region` from the State→Region reference table. Using the reference table for *every* row also catches a region that is spelled correctly but wrong for its state.

In [3]:
before = df["Region"].value_counts(dropna=False)
df = dc.standardise_region(df, log)
print(log.entries[-1]["decision"])
df["Region"].value_counts(dropna=False)

Fixed casing/typos; rebuilt 35 missing and 0 inconsistent regions from State (each state maps to one region).


Region
West       3849
East       3140
Central    2565
South      2013
Name: count, dtype: int64

## Step 3 — Standardise `Category`
**Problem:** `Tech`, `FURNITURE`, `office supplies`.
**Decision:** fix spelling, then re-derive `Category` from `Sub-Category` — each sub-category belongs to exactly one category (Phones → Technology), so this is a reliable source of truth.

In [4]:
df = dc.standardise_category(df, log)
df["Category"].value_counts()

Category
Office Supplies    6850
Furniture          2485
Technology         2232
Name: count, dtype: int64

## Step 4 — Fix numeric data types
**Problem:** `Sales` is text because 30 values look like `"$1,277.10"`.
**Decision:** remove `$` and `,` then convert. These are real sales, so deletion would understate revenue.

In [5]:
df = dc.fix_numeric_types(df, log)
print("Converted values:", log.entries[-1]["values_changed"])
print("Sales dtype now :", df["Sales"].dtype)

Converted values: 30
Sales dtype now : float64


## Step 5 — Remove exact duplicate rows
**Problem:** 70 rows are perfect copies of another row (a double export).
**Decision:** keep the first copy, drop the rest. **Repeated `Order ID`s are *not* touched** — they are multi-product orders (see notebook 01, Part 5b).

*Why now and not first?* Because steps 1–4 made formatting identical, any duplicate that differed only in spacing or `$` formatting is now detectable.

In [6]:
df = dc.remove_exact_duplicates(df, log)
e = log.entries[-1]
print(f"{e['rows_before']:,} -> {e['rows_after']:,} rows ({e['rows_removed']} removed)")

11,567 -> 11,497 rows (70 removed)


## Step 6 — Fix discounts typed as percentages
**Problem:** 18 rows have `Discount` like `20` or `40`.
**Reasoning:** RetailCo only uses fixed discount levels (0, 10, 15, 20, 30 … 80%). `20 / 100 = 0.20` is a valid level, so the intended value is unambiguous.
**Decision:** divide by 100 *only when the result is a valid level*; remove anything still invalid (none here).

In [7]:
df = dc.fix_discounts(df, log)
print("Corrected:", log.entries[-1]["values_changed"], "| max discount now:", df["Discount"].max())

Corrected: 18 | max discount now: 0.8


## Step 7 — Fill missing names from IDs
**Decision:** `Customer ID` and `Product ID` are complete. Look up the name used for the same ID on other rows. This is a *lookup*, not a guess.

In [8]:
df = dc.fill_names_from_ids(df, log)
print("Names filled:", log.entries[-1]["values_changed"])
print("Still missing:", int(df[["Customer Name", "Product Name"]].isna().sum().sum()))

Names filled: 45
Still missing: 0


## Step 8 — Dates
**Order Date** drives every time analysis (monthly trend, YoY growth). An order with no valid date cannot be placed in time, so it is **removed** (5 rows).

**Ship Date before Order Date** is impossible — but the *sale* is still real and its money is correct. Deleting the row would understate sales to fix a shipping-data problem. **Decision:** keep the row, set only `Ship Date` to missing. It is then excluded from the shipping-time analysis only.

In [9]:
df = dc.fix_dates(df, log)
for e in log.entries[-2:]:
    print(f"{e['step']:<30} removed={e['rows_removed']}  changed={e['values_changed']}")

Remove invalid order dates     removed=5  changed=5
Blank impossible ship dates    removed=0  changed=6


## Step 9 — Missing `Sales` or `Profit`
**Options considered:**

| Option | Verdict |
|---|---|
| Fill with the column mean/median | ❌ Invents revenue; a $5 pen could be given $380 of sales |
| Fill with the product's average | ❌ Still a guess, and Profit depends on the discount actually given |
| **Remove and record** | ✅ Only 17 rows (≈0.15%) — too few to change any conclusion, and nothing is invented |

In [10]:
df = dc.remove_missing_money(df, log)
print("Removed:", log.entries[-1]["rows_removed"])

Removed: 17


## Step 10 — Impossible values
`Sales <= 0` or `Quantity <= 0` cannot describe a sale. Returns are normally stored in a separate returns table, so these are entry errors → removed.
**Negative `Profit` is kept** — a sale at a deep discount can genuinely lose money, and those losses are one of the most important findings of the project.

In [11]:
df = dc.remove_invalid_values(df, log)
print("Removed:", log.entries[-1]["rows_removed"])
print("Loss-making lines kept:", int((df["Profit"] < 0).sum()))

Removed: 14
Loss-making lines kept: 1907


## Step 11 — Duplicate check on the true key (`Order ID` + `Product ID`)

In [12]:
df = dc.check_duplicate_order_lines(df, log)
print(log.entries[-1]["decision"])

Repeated Order IDs are normal (multi-product orders); 0 repeated Order ID + Product ID pairs found.


## Step 12 — Outliers: flag, don't delete
**IQR rule:** IQR = Q3 − Q1 (the spread of the middle 50%). A value above **Q3 + 1.5 × IQR** is a statistical outlier.

But *statistically unusual* is not the same as *wrong*. Let's see what the outliers actually are before deciding.

In [13]:
df = dc.flag_outliers(df, log)
print(log.entries[-1]["decision"])
hv = df[df["Is High Value"]]
print(f"\nOutlier lines: {len(hv):,} ({len(hv)/len(df):.1%} of lines) "
      f"carrying {hv['Sales'].sum()/df['Sales'].sum():.1%} of all sales")
hv.groupby("Sub-Category")["Sales"].agg(["count", "median", "max"]).sort_values("count", ascending=False).head(8).round(0)

Flagged lines with Sales > 1,170.48 (IQR rule). Kept: they are real large orders.

Outlier lines: 1,016 (8.9% of lines) carrying 42.4% of all sales


,count,median,max
Sub-Category,,,
Phones,268,1611.0,3600.0
Chairs,212,1612.0,2774.0
Copiers,125,2270.0,5105.0
Tables,120,1728.0,3360.0
Machines,95,1978.0,4814.0
Appliances,76,1520.0,2280.0
Bookcases,60,1555.0,2333.0
Accessories,34,1258.0,1436.0


**Decision: keep them.** The "outliers" are copiers, phones, chairs and tables — expensive products bought in normal quantities. Together they carry ~42% of revenue. Removing them would make Technology and Furniture look far smaller than they are. The `Is High Value` flag lets anyone filter them if a specific analysis needs it.

---
## Final validation
`dc.validate()` asserts the business rules. If any rule fails, the code stops with an error instead of silently producing wrong numbers.

In [14]:
dc.validate(df)
log.add("Final clean data", len(df), len(df), 0, "All validation checks passed.")
print("All validation checks passed ✔")

All validation checks passed ✔


## Before / after report

In [15]:
report = log.to_frame()
report[["step", "rows_before", "rows_after", "rows_removed", "values_changed", "decision"]]

,step,rows_before,rows_after,rows_removed,values_changed,decision
0,Load raw data,11567,11567,0,0,Starting point.
1,Strip whitespace,11567,11567,0,67,Trimmed spaces in text columns so identical values group together.
2,Standardise Region,11567,11567,0,81,Fixed casing/typos; rebuilt 35 missing and 0 inconsistent regions from State (each state maps to one region).
3,Standardise Category,11567,11567,0,43,Fixed casing/abbreviations; Category re-derived from Sub-Category.
4,Fix numeric types,11567,11567,0,30,"Removed '$' and ',' from Sales text and converted numeric columns."
5,Remove exact duplicates,11567,11497,70,70,Dropped fully identical rows (double export). Kept first copy.
6,Fix discounts,11497,11497,0,18,Converted percentage-style discounts (20 -> 0.20); removed values outside the allowed discount levels.
7,Fill names from IDs,11497,11497,0,45,Rebuilt missing Customer/Product names using their IDs.
8,Remove invalid order dates,11497,11492,5,5,Dropped rows whose Order Date is not a real date (e.g. 2023-02-30).
9,Blank impossible ship dates,11492,11492,0,6,Ship Date before Order Date set to missing; sale itself kept.


In [16]:
raw_rows, final_rows = len(raw), len(df)
removed = report.set_index("step")["rows_removed"]
summary = pd.Series({
    "Raw rows": raw_rows,
    "Exact duplicates removed": removed["Remove exact duplicates"],
    "Invalid order dates removed": removed["Remove invalid order dates"],
    "Missing Sales/Profit removed": removed["Remove missing Sales/Profit"],
    "Impossible Sales/Quantity removed": removed["Remove impossible Sales/Quantity"],
    "Values repaired (not removed)": int(dc.count_repaired_values(raw, df).sum()),
    "Final rows": final_rows,
    "Rows kept %": round(final_rows / raw_rows * 100, 2),
})
summary.to_frame("value")

,value
Raw rows,11567.00
Exact duplicates removed,70.00
Invalid order dates removed,5.00
Missing Sales/Profit removed,17.00
Impossible Sales/Quantity removed,14.00
Values repaired (not removed),275.00
Final rows,11461.00
Rows kept %,99.08


**Why "values repaired" is not the sum of the log's `values_changed` column:** the log counts changes *per step*, so a value like `" east "` is counted twice (trimmed in step 1, re-cased in step 2), and repairs on rows removed later are still counted. `count_repaired_values()` compares the final clean data with the raw export cell by cell, so each repaired value is counted exactly once:

In [17]:
dc.count_repaired_values(raw, df).to_frame("values repaired")

,values repaired
Ship Date,6
Customer Name,25
City,40
Region,93
Category,44
Product Name,20
Sales,29
Discount,18


**How do we know cleaning didn't remove useful information?**
* Of the 106 rows removed, 70 were exact copies (no information lost at all).
* The other 36 (0.3%) had a missing/impossible value that cannot be rebuilt.
* 275 problem values were **repaired instead of deleted**.
* `tests/test_pipeline.py` compares the cleaned data against the generator's error-free version and confirms every repaired value matches the truth.

---
## Feature engineering

New columns calculated from existing ones (logic in `src/feature_engineering.py`):

| Feature | Formula | Why it is useful |
|---|---|---|
| `Year`, `Quarter`, `Month`, `Month Number`, `Year-Month` | parts of `Order Date` | Group by time without re-parsing dates |
| `Shipping Days` | `Ship Date − Order Date` | Delivery speed by ship mode |
| `Profit Margin %` | `Profit / Sales × 100` | Profit per $ of sales — comparable across sizes |
| `Unit Price` | `Sales / Quantity` | Average price actually paid per unit |
| `Gross Sales` | `Sales / (1 − Discount)` | Revenue at full list price |
| `Discount Amount` | `Gross Sales − Sales` | Revenue given away by discounting |
| `Is Loss` | `Profit < 0` | Quick filter for loss-making lines |
| `Discount Band` | groups of `Discount` | Readable buckets for charts |

### Why Profit Margin beats Profit alone
Region A: $1,000,000 sales, $50,000 profit → **5%** margin.
Region B: $300,000 sales, $45,000 profit → **15%** margin.
A has more profit *only because it is bigger*. B keeps three times as much from every dollar. Margin removes the size effect, so regions of different sizes can be compared fairly.

⚠️ **Never average `Profit Margin %` across rows.** A region's margin is `SUM(Profit) / SUM(Sales)`. Averaging row margins would give a $3 pen the same weight as a $3,000 copier.

In [18]:
df = engineer_features(df)
df[["Order Date", "Year", "Quarter", "Year-Month", "Sales", "Discount", "Profit",
    "Profit Margin %", "Unit Price", "Gross Sales", "Discount Amount", "Shipping Days", "Discount Band"]].head()

,Order Date,Year,Quarter,Year-Month,Sales,Discount,Profit,Profit Margin %,Unit Price,Gross Sales,Discount Amount,Shipping Days,Discount Band
0,2024-08-07,2024,Q3,2024-08,71.02,0.3,17.93,25.25,11.84,101.46,30.44,5,21-40%
1,2024-03-24,2024,Q1,2024-03,17.71,0.1,4.36,24.62,5.90,19.68,1.97,2,1-20%
2,2022-11-08,2022,Q4,2022-11,4338.15,0.0,1748.86,40.31,1446.05,4338.15,0.00,0,No discount
3,2022-12-02,2022,Q4,2022-12,27.86,0.2,9.43,33.85,9.29,34.82,6.96,5,1-20%
4,2024-07-19,2024,Q3,2024-07,202.80,0.0,73.07,36.03,20.28,202.80,0.00,5,No discount


**Demonstration — why averaging row margins is wrong:**

In [19]:
correct = df["Profit"].sum() / df["Sales"].sum() * 100
wrong = df["Profit Margin %"].mean()
print(f"Correct company margin  SUM(Profit)/SUM(Sales): {correct:6.2f}%")
print(f"Wrong   average of row-level margins          : {wrong:6.2f}%")

Correct company margin  SUM(Profit)/SUM(Sales):  10.29%
Wrong   average of row-level margins          :  11.74%


The simple average is **wrong by about 1.5 percentage points**. It treats every line as equally important, so thousands of small, high-margin lines (paper, labels, binders) outweigh the fewer large, low-margin lines (tables, phones). The weighted formula counts each dollar once, which is what "margin" means to a finance team.

## Save the outputs

In [20]:
CLEAN_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_FILE, index=False, date_format="%Y-%m-%d", lineterminator="\n")
report.to_csv(CLEANING_LOG_FILE, index=False, lineterminator="\n")
print(f"Saved {CLEAN_FILE.name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

Saved sales_cleaned.csv: 11,461 rows x 32 columns


**Next:** `03_eda.ipynb` explores the clean data.